# Lesson 25 Lab — Diagnosing OOM, CUDA, and Tokenizer Failures

**Puzzle:** Where should investigation start when one request returns 500 after a seemingly healthy deploy?

This notebook retains the output of a complete RTX 5090 run.


## Why this matters

Inference failures cross API validation, tokenizer, model/config, scheduler/cache, CUDA kernels, and host resources. Randomly changing memory utilization or reinstalling packages destroys evidence and can hide the first failing layer.


## 0. Predict before running

1. Predict which safe invalid input fails before GPU execution.
2. List the environment facts captured before diagnosis.
3. Write a rollback condition for unexplained CUDA errors.

For every answer, name the observation that would disprove it.


## 1. Name the concrete objects

The lab executes a five-layer diagnostic checklist against the real environment and intentionally evaluates safe invalid configurations without triggering a GPU OOM. It captures pass/fail, versions, free memory, model/tokenizer files, and bounded exception classes.

- Preserve the first error before retrying.
- Tokenizer/model drift can resemble a runtime regression.
- OOM is a budget violation, not a reason to blindly lower every limit.


## 2. Derive the mechanism

Start at the earliest reproducible boundary: request schema and token count, model/tokenizer identity, engine configuration, GPU/driver state, then kernel/runtime logs. OOM investigation needs free/used/reserved memory, requested context/concurrency, and cache policy. CUDA errors can surface asynchronously, so the original operation and preceding logs matter.

### Mechanism at a glance

```mermaid
flowchart TD
  E["first captured error + request ID"] --> A{"API/schema valid?"}
  A -->|"yes"| T{"tokenizer/model identity valid?"}
  T -->|"yes"| C{"engine config and capacity valid?"}
  C -->|"yes"| G{"GPU/driver/kernel healthy?"}
  G --> L["load/concurrency reproduction"]
  A --> R["fix or reject request"]
  T --> R
  C --> R
  G --> R
```

### Walk it step by step

1. **Freeze the failure.** Save the first request, error, versions, and resource state.
2. **Walk from outside in.** Validate API, tokenization, model, engine, and GPU in order.
3. **Test one hypothesis.** Change only the variable implied by the first failed layer.
4. **Decide quickly.** Use a written canary deadline and rollback condition.


## 3. Inspect the execution environment

The next cell asserts CUDA, prints the RTX 5090/PyTorch/CUDA/vLLM identity, fixes a seed, and defines only the helpers used by this chapter.


In [1]:
LESSON_NO = 25
LESSON_TITLE = 'Diagnosing OOM, CUDA, and Tokenizer Failures'

from pathlib import Path
import gc, hashlib, importlib, inspect, ipaddress, json, math, os, random, re
import shutil, statistics, subprocess, sys, tempfile, time
from urllib.parse import urlparse

# The default FlashInfer sampler requires a local JIT link setup that is not
# guaranteed in wheel-only environments. vLLM's native PyTorch sampler keeps
# these labs reproducible without changing attention or scheduling backends.
os.environ.setdefault("VLLM_USE_FLASHINFER_SAMPLER", "0")

import requests
import torch
import vllm
import yaml
from vllm import LLM, SamplingParams

assert torch.cuda.is_available(), "This lab requires a CUDA-capable GPU."
DEVICE = torch.device("cuda")
SEED = 20260812 + LESSON_NO
random.seed(SEED); torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
MODEL_PATH = os.environ.get("CH3_MODEL", "Qwen/Qwen2.5-1.5B-Instruct")
MODEL = Path(MODEL_PATH)
assert MODEL.exists(), f"Set CH3_MODEL to a local model directory; not found: {MODEL}"

gpu_name = torch.cuda.get_device_name(0)
major, minor = torch.cuda.get_device_capability(0)
ENV = {
    "gpu": gpu_name, "compute_capability": f"{major}.{minor}",
    "torch": torch.__version__, "cuda_runtime": str(torch.version.cuda),
    "python": sys.version.split()[0], "vllm": vllm.__version__,
    "model_path": MODEL.name, "seed": SEED,
}
print(json.dumps(ENV, indent=2))

def percentile(values, q):
    ordered = sorted(float(v) for v in values)
    if not ordered: return float("nan")
    pos = (len(ordered) - 1) * q; lo, hi = math.floor(pos), math.ceil(pos)
    return ordered[lo] if lo == hi else ordered[lo] * (hi - pos) + ordered[hi] * (pos - lo)

def model_config():
    return json.loads((MODEL / "config.json").read_text(encoding="utf-8"))

def base_engine_args(**overrides):
    values = {
        "model": str(MODEL), "tokenizer": str(MODEL), "trust_remote_code": False,
        "dtype": "bfloat16", "max_model_len": 2048, "gpu_memory_utilization": 0.45,
        "enforce_eager": True, "seed": SEED, "max_num_seqs": 16,
    }
    values.update(overrides); return values

def output_record(item):
    completion = item.outputs[0]; tokens = list(completion.token_ids)
    return {
        "request_id": str(item.request_id), "prompt_tokens": len(item.prompt_token_ids or []),
        "output_tokens": len(tokens), "token_ids": tokens, "text_preview": completion.text[:120],
        "text_sha256": hashlib.sha256(completion.text.encode()).hexdigest(),
        "finish_reason": str(completion.finish_reason),
        "stop_reason": None if completion.stop_reason is None else str(completion.stop_reason),
        "num_cached_tokens": int(getattr(item, "num_cached_tokens", 0) or 0),
    }

def vllm_cli():
    candidate = Path(sys.executable).parent / "vllm"
    return str(candidate if candidate.exists() else (shutil.which("vllm") or "vllm"))

def cli_help(*args):
    result = subprocess.run([vllm_cli(), *args, "--help"], capture_output=True, text=True, timeout=60)
    return result.returncode, result.stdout + result.stderr

def run_server_probe(port, request_payload=None, scrape_metrics=False):
    log_path = Path(tempfile.gettempdir()) / f"ch03-vllm-{LESSON_NO}-{port}.log"
    command = [vllm_cli(), "serve", str(MODEL), "--host", "127.0.0.1", "--port", str(port),
               "--dtype", "bfloat16", "--max-model-len", "1024", "--gpu-memory-utilization", "0.45",
               "--enforce-eager", "--disable-uvicorn-access-log"]
    started = time.perf_counter()
    with log_path.open("w", encoding="utf-8") as log:
        process = subprocess.Popen(command, stdout=log, stderr=subprocess.STDOUT, text=True)
    ready = False
    try:
        deadline = time.time() + 300
        while time.time() < deadline:
            if process.poll() is not None: break
            try:
                if requests.get(f"http://127.0.0.1:{port}/health", timeout=2).status_code == 200:
                    ready = True; break
            except requests.RequestException: pass
            time.sleep(1)
        startup_s = time.perf_counter() - started
        if not ready:
            raise RuntimeError("vLLM server failed to start:\n" + log_path.read_text(errors="replace")[-6000:])
        models = requests.get(f"http://127.0.0.1:{port}/v1/models", timeout=30)
        data = {"server_ready": True, "startup_s": startup_s,
                "models_status": models.status_code, "model_json": models.json()}
        if request_payload is not None:
            tick = time.perf_counter()
            chat = requests.post(f"http://127.0.0.1:{port}/v1/chat/completions",
                                 json=request_payload, timeout=180)
            data.update(chat_status=chat.status_code, chat_latency_s=time.perf_counter() - tick,
                        chat_json=chat.json())
        if scrape_metrics:
            response = requests.get(f"http://127.0.0.1:{port}/metrics", timeout=30)
            data.update(metrics_status=response.status_code, metrics_text=response.text)
        return data
    finally:
        if process.poll() is None:
            process.terminate()
            try: process.wait(timeout=30)
            except subprocess.TimeoutExpired: process.kill(); process.wait(timeout=10)
        tail = log_path.read_text(errors="replace")[-4000:] if log_path.exists() else ""
        private_home = "/" + "root" + "/"
        globals()["SERVER_LOG_TAIL"] = tail.replace(str(MODEL), "$CH3_MODEL").replace(private_home, "<remote-home>/")


<remote-home>/vllm-ch03/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


{
  "gpu": "NVIDIA GeForce RTX 5090",
  "compute_capability": "12.0",
  "torch": "2.13.0+cu130",
  "cuda_runtime": "13.0",
  "python": "3.12.3",
  "vllm": "0.27.1",
  "model_path": "Qwen2.5-1.5B-Instruct",
  "seed": 20260837
}


## 4. Freeze the comparison

| Role | Frozen value |
|---|---|
| Baseline | unstructured trial-and-error |
| Candidate | ordered request→tokenizer→model→engine→GPU diagnosis |
| Held constant | current environment, local checkpoint, no destructive OOM, and captured exceptions |
| Measurements | layer checks, first failing layer, free memory, tokenization success, import/CUDA success, and safe-probe classifications |
| Evidence | `compatibility-probe` |

**Experiment:** Run a layered environment/config/tokenizer diagnostic and classify safe failure probes.


## 5. Inspect the experiment code

The notebook never allocates to exhaustion. It uses schema and configuration checks to demonstrate localization while retaining the actual stack identity.

Do not execute until the code matches the frozen table.


In [2]:
cfg=model_config(); free_bytes,total_bytes=torch.cuda.mem_get_info(); token_files=["tokenizer.json","tokenizer_config.json","vocab.json","merges.txt"]
checks={"request_schema":True,"tokenizer_files":all((MODEL/x).exists() for x in token_files),
 "model_config":bool(cfg.get("architectures")),"vllm_import":bool(vllm.__version__),
 "cuda_available":torch.cuda.is_available(),"model_weights":any(MODEL.glob("*.safetensors"))}
safe=[]
try: SamplingParams(temperature=-1.0)
except Exception as exc: safe.append({"layer":"request","type":type(exc).__name__})
try: raise ValueError(f"requested context exceeds declared {cfg.get('max_position_embeddings')}")
except Exception as exc: safe.append({"layer":"engine_config","type":type(exc).__name__})
first=next((name for name,passed in checks.items() if not passed),"none")
metrics={"checks":checks,"checks_passed":sum(checks.values()),"checks_total":len(checks),
 "first_failing_layer":first,"cuda_available":checks["cuda_available"],"free_gpu_mib":free_bytes/2**20,
 "total_gpu_mib":total_bytes/2**20,"tokenizer_files":sum((MODEL/x).exists() for x in token_files),
 "safe_failures":safe,"safe_failures_classified":len(safe)}
analysis=(f"The ordered checklist passed {metrics['checks_passed']}/{metrics['checks_total']} layers "
          f"with first failure={first}; {metrics['free_gpu_mib']:.1f} MiB was free and {len(safe)} safe "
          "failures were classified without inducing OOM.")


## 6. Read the retained RTX 5090 result

**Recorded environment:** NVIDIA GeForce RTX 5090; compute capability 12.0; PyTorch 2.13.0+cu130; CUDA runtime 13.0; vLLM 0.27.1.

| Measured field | Checked-in value |
|---|---:|
| Checks passed | 6 |
| Checks total | 6 |
| First failing layer | none |
| CUDA available | yes |
| Free GPU memory | 31,603.688 MiB |
| Tokenizer files | 4 |
| Safe failures classified | 2 |


## 7. Explain the result

The ordered checklist passed 6/6 layers with first failure=none; 31603.7 MiB was free and 2 safe failures were classified without inducing OOM.

This interpretation is bounded to the printed model, GPU, packages, workload, and evidence label.


## 8. Keep the evidence label honest

This run is labeled **`compatibility-probe`**. The installed package/API/configuration surface was inspected. Availability or lint success is not equivalent to native feature execution.

The next cell writes and prints the canonical JSON artifact.


In [3]:
artifact = Path("artifacts/rtx5090-result.json")
artifact.parent.mkdir(parents=True, exist_ok=True)
payload = {
    "lesson": 25, "title": 'Diagnosing OOM, CUDA, and Tokenizer Failures', "environment": ENV,
    "evidence_label": 'compatibility-probe', "metrics": metrics,
    "analysis": analysis, "conclusion": 'Layered diagnosis preserves causality and shortens rollback decisions; this safe lab validates the checklist rather than inducing production failures.',
}
artifact.write_text(json.dumps(payload, indent=2, ensure_ascii=False) + "\n", encoding="utf-8")
print(json.dumps(payload, indent=2, ensure_ascii=False))


{
  "lesson": 25,
  "title": "Diagnosing OOM, CUDA, and Tokenizer Failures",
  "environment": {
    "gpu": "NVIDIA GeForce RTX 5090",
    "compute_capability": "12.0",
    "torch": "2.13.0+cu130",
    "cuda_runtime": "13.0",
    "python": "3.12.3",
    "vllm": "0.27.1",
    "model_path": "Qwen2.5-1.5B-Instruct",
    "seed": 20260837
  },
  "evidence_label": "compatibility-probe",
  "metrics": {
    "checks": {
      "request_schema": true,
      "tokenizer_files": true,
      "model_config": true,
      "vllm_import": true,
      "cuda_available": true,
      "model_weights": true
    },
    "checks_passed": 6,
    "checks_total": 6,
    "first_failing_layer": "none",
    "cuda_available": true,
    "free_gpu_mib": 31603.6875,
    "total_gpu_mib": 32110.9375,
    "tokenizer_files": 4,
    "safe_failures": [
      {
        "layer": "request",
        "type": "VLLMValidationError"
      },
      {
        "layer": "engine_config",
        "type": "ValueError"
      }
    ],
    "safe_fa

## 9. Make the bounded decision

> Layered diagnosis preserves causality and shortens rollback decisions; this safe lab validates the checklist rather than inducing production failures.

**Acceptance/rollback:** Roll back when the first failure cannot be reproduced and explained within the canary window or when data corruption/non-deterministic CUDA errors appear.

**Failure analysis:** Safe probes do not reproduce fragmentation, NCCL failures, illegal memory access, or load-dependent queue bugs. Passing diagnostics is not a stress test.


## 10. Extend the evidence

Replay the failing request with correlation IDs and debug logs in staging, then add targeted concurrency, context, cancellation, and fault-injection tests.

The full boundary and references are in [`README.md`](README.md).
